# Prediccion del Mundial 2026 - Machine Learning + econometria

Notebook auto-actualizable: lee el Excel `Mundial_2026_fuente_datos.xlsx`,
toma los resultados ya cargados como hechos fijos y recalcula las
probabilidades cada vez que se reejecuta (Entorno de ejecucion -> Ejecutar todo).

Flujo profesional de prediccion:

1. Carga y limpieza de datos (48 selecciones, fixture, cuadro final).
2. Construccion de features por partido: rating Elo, ranking y puntos FIFA,
   titulos, apariciones, mejor resultado historico, valor de plantel, edad,
   **puntaje del DT**, **clasificatoria ponderada por confederacion** y
   **proporcion de jugadores en las 5 grandes ligas**.
3. **Auto-calibracion** de los parametros econometricos (nu, lambda_prior) por
   validacion out-of-fold.
4. Entrenamiento de un **zoo de modelos**: Elo probabilistico, Dixon-Coles/Poisson
   y varios modelos de ML con **auto-tuning** de hiperparametros (logit,
   RandomForest, ExtraTrees, GradientBoosting, HistGradientBoosting y, si estan
   disponibles, XGBoost y LightGBM), con calibracion de probabilidades.
5. Evaluacion out-of-fold (log-loss, accuracy, Brier) + **calibracion (ECE)** y
   seleccion de los **3 mejores modelos**; la prediccion 1/X/2 final es un
   **blend ponderado** de esos tres.
6. Simulacion Monte Carlo del torneo (Dixon-Coles como generador de marcadores)
   con desempate oficial FIFA, localia moderada a los anfitriones en
   eliminatorias y **resultados de eliminatorias ya jugados fijados** (los
   equipos eliminados quedan descartados).

Entrega probabilidades, no recomendaciones de apuestas.

## 1. Preparacion del entorno (clona el repo e instala dependencias en Colab)

In [ ]:
import os, sys

EN_COLAB = 'google.colab' in sys.modules
REPO = 'ML_prediccion_mundial2026'
REPO_URL = 'https://github.com/santiagoriverti/ML_prediccion_mundial2026.git'

# En Colab: clonar el repo (trae src/ y el Excel) e instalar requirements.
if EN_COLAB:
    if not os.path.exists(REPO):
        !git clone -q {REPO_URL}
    %cd {REPO}
    !pip install -q -r requirements.txt

# Localizar la carpeta src/ (funciona tanto en Colab como en local).
for cand in ['src', '../src', os.path.join(REPO, 'src')]:
    if os.path.exists(cand):
        sys.path.insert(0, os.path.abspath(cand))
        RAIZ = os.path.dirname(os.path.abspath(cand))
        break
print('Raiz del proyecto:', RAIZ)

## 2. Carga del Excel (siempre la ultima version commiteada)

In [ ]:
import glob, urllib.request
import warnings; warnings.filterwarnings('ignore')

# 1o intenta la raw URL del repo (toma el ultimo commit); 2o archivo local;
# 3o (solo Colab) subida manual.
RAW = ('https://raw.githubusercontent.com/santiagoriverti/'
       'ML_prediccion_mundial2026/main/Mundial_2026_fuente_datos.xlsx')
datos_bytes = None
try:
    datos_bytes = urllib.request.urlopen(RAW, timeout=25).read()
    print('OK - Excel cargado desde la raw URL del repo.')
except Exception as e:
    print('La raw URL fallo (', e, '); busco archivo local .xlsx...')
    locales = glob.glob(os.path.join(RAIZ, '*.xlsx')) + glob.glob('*.xlsx')
    if locales:
        datos_bytes = open(locales[0], 'rb').read()
        print('OK - Excel local:', locales[0])
    elif EN_COLAB:
        from google.colab import files
        subido = files.upload()
        datos_bytes = list(subido.values())[0]
        print('OK - Excel subido manualmente.')
    else:
        raise FileNotFoundError('No se encontro el Excel insumo.')

## 3. Lectura y limpieza de datos

Se cargan las 48 selecciones, el fixture de grupos y el cuadro de eliminatorias.
El loader tolera hojas plantilla vacias, normaliza nombres por `Pais` y descarta
la fila de nota al pie.

In [ ]:
from data_loader import cargar_datos
from features import imputar_rating_base, construir_dataset_partidos

datos = cargar_datos(datos_bytes)
print('Equipos:', datos.meta['n_equipos'],
      '| Partidos de grupo:', datos.meta['n_partidos_grupo'],
      '| Ya jugados:', datos.meta['n_jugados'],
      '| Sin puntos FIFA (imputados):', datos.meta['n_sin_puntos'])

equipos = imputar_rating_base(datos.equipos)
equipos[['pais', 'grupo', 'confederacion', 'rating_base', 'rating_imputado']] \
    .sort_values('rating_base', ascending=False).head(10)

## 4. Actualizacion del Elo con los resultados ya cargados

Antes de simular, el rating de cada seleccion se mueve con los partidos ya
jugados (los hechos fijan el pronostico), con factor de margen de victoria.

In [ ]:
from simulate import actualizar_elo

equipos = actualizar_elo(equipos, datos.fixture, K=32.0)
dataset = construir_dataset_partidos(equipos, datos.fixture)
print('Dataset por partido:', dataset.shape, '| con resultado (jugados):',
      int(dataset['jugado'].sum()))
equipos[['pais', 'rating_base']].sort_values('rating_base', ascending=False).head(8)

## 5. Variables (features) que usa el modelo

Cada partido se describe por la diferencia A-B de los atributos de las dos
selecciones. Estas son las variables que entran a los modelos supervisados:

In [ ]:
from features import COLUMNAS_FEATURES
import pandas as pd

descripcion = {
    'd_rating': 'Diferencia de rating Elo (Puntos FIFA + actualizacion por resultados)',
    'd_ranking': 'Diferencia de ranking FIFA (signo invertido: mejor = menor)',
    'd_puntos': 'Diferencia de Puntos FIFA',
    'd_titulos': 'Diferencia de titulos mundiales',
    'd_apariciones': 'Diferencia de apariciones en Mundiales',
    'd_mejor_result': 'Diferencia de mejor resultado historico (ordinal)',
    'd_valor_plantel': 'Diferencia de valor de plantel (Transfermarkt, decenas de EUR M)',
    'd_edad': 'Diferencia de edad promedio del plantel',
    'd_dt': 'Diferencia de puntaje de trayectoria del DT (seleccion + clubes, 0-100)',
    'd_clasif': 'Diferencia de puntaje de clasificatoria ponderado por dificultad de confederacion',
    'd_top5': 'Diferencia de proporcion de jugadores en las 5 grandes ligas',
    'anfitrion': 'Ventaja de localia (1 si A es sede, -1 si B, 0 si no)',
    'altitud': 'Altitud de la sede (placeholder; hoy 0)',
}
pd.DataFrame({'feature': COLUMNAS_FEATURES,
              'descripcion': [descripcion.get(c, '') for c in COLUMNAS_FEATURES]})

## 6. Auto-calibracion de parametros + entrenamiento del zoo de modelos

Primero se **auto-calibran** los parametros econometricos (`nu` del Elo y
`lambda_prior` de Dixon-Coles) buscando, por validacion out-of-fold, los valores
que minimizan el log-loss (sin afinar a mano).

Luego se entrena:
- **Dixon-Coles/Poisson** (con el `lambda_prior` calibrado): genera marcadores
  para la simulacion del torneo.
- Un **zoo de modelos de ML** para 1/X/2 con **auto-tuning** de hiperparametros
  (`RandomizedSearchCV`) y calibracion: regresion logistica, RandomForest,
  ExtraTrees, GradientBoosting, HistGradientBoosting y, si estan instalados,
  **XGBoost** y **LightGBM**.

Con muestra chica (pocos partidos jugados) el nucleo robusto sigue siendo
Elo + Dixon-Coles; el ML entra como complemento y la seleccion del paso 7 decide
de forma data-driven cuanto pesa cada uno.

In [ ]:
from models import DixonColes, entrenar_modelos_ml, calibrar_parametros

# 1) Auto-calibracion de parametros econometricos (out-of-fold).
cal = calibrar_parametros(dataset, equipos)
NU, LAMBDA = cal['nu'], cal['lambda_prior']
print(f"Parametros auto-calibrados -> nu={NU} | lambda_prior={LAMBDA} "
      f"| log-loss OOF={cal['log_loss']}")

# 2) Dixon-Coles (generador de marcadores) con el lambda calibrado.
dc = DixonColes(equipos, lambda_prior=LAMBDA).entrenar(dataset)

# 3) Zoo de modelos ML con auto-tuning de hiperparametros (una sola vez).
modelos_ml, reporte = entrenar_modelos_ml(dataset, tune=True)
print('\nModelos ML entrenados:', list(modelos_ml.keys()))
print('Log-loss CV por modelo:', {k: round(v, 3) for k, v in reporte['cv'].items()})
print('Hiperparametros elegidos (auto-tuning):')
for k, v in reporte['hiperparams'].items():
    print('   ', k, '->', v)

## 7. Evaluacion out-of-fold y seleccion de los 3 mejores modelos

Se compara cada modelo 1/X/2 (Elo, Dixon-Coles y todo el zoo de ML) por
validacion cruzada **out-of-fold** sobre los partidos ya jugados: en cada fold se
reentrenan Dixon-Coles y los modelos ML solo con el train (con los hiperparametros
ya elegidos) y se predice el test, sin fuga de informacion.

La **prediccion final** se hace con los **3 mejores modelos** (menor log-loss),
combinados en un **blend ponderado** (mas peso al de menor log-loss). Asi no se
apuesta todo a un solo modelo: se promedian los tres mas confiables.

Nota: la simulacion del torneo necesita un generador de marcadores, rol que
cumple Dixon-Coles; la seleccion top-3 aplica al pronostico 1/X/2.

In [ ]:
from models import evaluar_modelos, seleccionar_top

# Evaluacion OOF (reusa los hiperparametros ya buscados y los parametros
# econometricos calibrados). devolver_oof=True entrega las predicciones OOF
# (oof) y los resultados reales (y_eval) para medir la calibracion mas abajo.
tabla_eval, mejor, oof, y_eval = evaluar_modelos(
    dataset, equipos, devolver_oof=True, nu=NU, lambda_prior=LAMBDA,
    hiperparams=reporte['hiperparams'])

# Los 3 mejores modelos base (excluye 'ensemble') y sus pesos para el blend.
top3, pesos_top = seleccionar_top(tabla_eval, k=3)
print('Los 3 mejores modelos (prediccion final):', top3)
print('Pesos del blend:', {k: round(v, 3) for k, v in pesos_top.items()})
tabla_eval

## 8. Pronostico por partido pendiente (blend de los 3 mejores)

Para cada partido aun no jugado: P(1/X/2) del **blend de los 3 mejores modelos**,
goles esperados (Dixon-Coles) y marcador mas probable.

In [ ]:
from models import pronostico_partidos

tabla_partidos = pronostico_partidos(dataset, equipos, dc, modelos_ml,
                                     modelos_top=top3, pesos_top=pesos_top, nu=NU)
print('Partidos pendientes de jugar:', len(tabla_partidos),
      '| prediccion = blend top-3:', top3)
tabla_partidos.head(20)

## 8. Pronostico por partido pendiente (con el mejor modelo)

Para cada partido aun no jugado: P(1/X/2) del mejor modelo seleccionado, goles
esperados (Dixon-Coles) y marcador mas probable.

In [ ]:
from models import pronostico_partidos

tabla_partidos = pronostico_partidos(dataset, equipos, dc, modelos_ml,
                                     modelo=mejor_modelo)
print('Partidos pendientes de jugar:', len(tabla_partidos),
      '| modelo usado:', mejor_modelo)
tabla_partidos.head(20)

## 9. Simulacion Monte Carlo del torneo

Completa los partidos no jugados, resuelve los grupos con el desempate oficial
FIFA, asigna los 8 mejores terceros, arma el cuadro y simula hasta la final
(prorroga/penales por fuerza). Los anfitriones reciben localia moderada en
eliminatorias. Los partidos ya jugados quedan fijos.

In [ ]:
from simulate import simular_torneo

N_SIMS = 20000
resultados = simular_torneo(equipos, datos.fixture, datos.bracket, dc,
                            n_sims=N_SIMS, semilla=2026)
print('Corridas:', resultados['n_sims'])

## 10. Ranking de probabilidad de ser campeon

In [ ]:
tabla_campeon = resultados['campeon'].copy()
tabla_campeon['prob_campeon_%'] = (tabla_campeon['prob_campeon'] * 100).round(2)
tabla_campeon[['pais', 'prob_campeon_%']].head(20)

## 11. Probabilidad de alcanzar cada ronda

In [ ]:
cols_ronda = ['pais', 'prob_32avos', 'prob_16avos', 'prob_Cuartos',
              'prob_Semifinales', 'prob_Final', 'prob_campeon']
av = resultados['avance'][cols_ronda].copy()
for c in cols_ronda[1:]:
    av[c] = (av[c] * 100).round(1)
av.head(16)

## 12. Cuadro de eliminatorias del escenario mas probable

Escenario unico y consistente (cada seleccion aparece una sola vez): completa
los partidos de grupo pendientes con su marcador esperado y resuelve el cuadro
de 32avos con nombres de seleccion. Es una proyeccion que cambia al recargar
resultados; la simulacion Monte Carlo de arriba mantiene toda la incertidumbre.

In [ ]:
from viz import grafico_campeon, heatmap_avance, grafico_calibracion

ruta1 = grafico_campeon(resultados['campeon'], top=15)
ruta2 = heatmap_avance(resultados['avance'], top=20)
ruta3 = grafico_calibracion(tabla_calib, ece, 'blend top-3 ' + '+'.join(top3))
print('Guardado en:', ruta1, ',', ruta2, 'y', ruta3)

## 13. Graficos (se guardan en `outputs/`)

In [ ]:
from viz import grafico_campeon, heatmap_avance

ruta1 = grafico_campeon(resultados['campeon'], top=15)
ruta2 = heatmap_avance(resultados['avance'], top=20)
print('Guardado en:', ruta1, 'y', ruta2)

## 14. Probabilidades por grupo (gana grupo / clasifica)

In [ ]:
os.makedirs('outputs', exist_ok=True)
tabla_partidos.to_csv('outputs/pronostico_partidos.csv', index=False)
tabla_eval.to_csv('outputs/evaluacion_modelos.csv', index=False)
resumen_calib.to_csv('outputs/calibracion_ece.csv', index=False)
tabla_calib.to_csv('outputs/calibracion_reliability.csv', index=False)
resultados['campeon'].to_csv('outputs/prob_campeon.csv', index=False)
resultados['avance'].to_csv('outputs/prob_avance.csv', index=False)
resultados['grupos'].to_csv('outputs/prob_grupos.csv', index=False)
bracket.to_csv('outputs/bracket_proyectado.csv', index=False)

# Resumen del modelo elegido (parametros calibrados + top-3 + pesos).
import json
with open('outputs/resumen_modelo.json', 'w', encoding='utf-8') as fh:
    json.dump({'nu': NU, 'lambda_prior': LAMBDA,
               'top3': top3, 'pesos_top': pesos_top,
               'hiperparams': reporte['hiperparams']},
              fh, ensure_ascii=False, indent=2, default=float)
print('CSVs y resumen del modelo guardados en outputs/.')

## 15. Guardar resultados a `outputs/`

In [ ]:
os.makedirs('outputs', exist_ok=True)
tabla_partidos.to_csv('outputs/pronostico_partidos.csv', index=False)
tabla_eval.to_csv('outputs/evaluacion_modelos.csv', index=False)
resultados['campeon'].to_csv('outputs/prob_campeon.csv', index=False)
resultados['avance'].to_csv('outputs/prob_avance.csv', index=False)
resultados['grupos'].to_csv('outputs/prob_grupos.csv', index=False)
bracket.to_csv('outputs/bracket_proyectado.csv', index=False)
print('CSVs de salida guardados en outputs/.')

---
## Como actualizar el pronostico

1. Abri `Mundial_2026_fuente_datos.xlsx` y carga los goles de los partidos
   jugados en la hoja Fixture_Grupos (columnas Goles A / Goles B) y/o en
   Eliminatorias.
2. Commitea y pushea el Excel al repo (la raw URL siempre toma el ultimo commit).
3. Reejecuta el notebook (Ejecutar todo). El rating se actualiza con los nuevos
   resultados, se reentrenan y reevaluan los modelos y los partidos pendientes
   se vuelven a simular.